## Exercise 05.1 (random numbers)

- Using the `randint` function from the `random` module (https://docs.python.org/3/library/random.html#random.randint) to
  develop a function `dice_roll` that emulates the roll of a dice with $n$ sides. The number of sides `n` should an argument to the function.

- For $n=6$, devise and implement a test to check that it is a fair dice.

#### (a) Dice roll code:

In [ ]:
import random

# implementing dice roll code 
def dice_roll(n): 

    # return a face at random between smallest (1) and biggest (n) values 
    face = random.randint(1, n)
    return face 

In [33]:
## tests ##
for n in range(1, 20):
    for j in range(100):
        value = dice_roll(n) 
        assert value >= 1 and value <= n

#### (b) Test for fairness

In [168]:
# To check for fairness, we roll the dice a large number of times and check how many times we get each value. 
# We expect to get each value close to $1/6$ of the times. 

import random

# implementing dice roll code 
def dice_roll(n): 

    # return a face at random between smallest (1) and biggest (n) values 
    face = random.randint(1, n)
    return face 

ones = 0
twos = 0
threes = 0
fours = 0
fives = 0
sixes = 0

for n in range(100):
    value = dice_roll(6)

    if value == 1:
        ones += 1
    elif value == 2:
        twos += 1
    elif value == 3:
        threes += 1
    elif value == 4:
        fours += 1
    elif value == 5:
        fives += 1
    elif value == 6:
        sixes += 1

outcomes = [ones, twos, threes, fours, fives, sixes]
print(outcomes)

[20, 16, 21, 13, 16, 14]


In [169]:
## tests: testing for fairness: each probability is ~1/6 ##

for num in outcomes:

    prob = num/100
    assert 0.09 <= prob <= 0.23
    print(prob)

0.2
0.16
0.21
0.13
0.16
0.14


## Exercise 05.2 (variance estimation)

For a random variable $X$, the variance of $X$ is defined as  

$$
\begin{align}
\mathrm{Var}\left[ X \right] &= \mathrm{E}\left[\left( X - \mu \right)^2\right]  \\ 
&= \mathrm{E}\left[ X^{2} \right] - \mathrm{E}\left[ X \right]^2
\end{align}
$$

where $\mathrm{E}$ is the 'expectation' (mean of something) and $\mu = \mathrm{E}(X)$ is the mean of $X$. If we have all data (the entire 'population'),
the variance can be computed from:

$$
\mathrm{Var}\left[ X \right] = \frac{\sum_{i=0}^{N-1} x^{2}_{i}}{N} - \left( \frac{ \left( \sum_{i=0}^{N-1} x_{i} \right)}{N} \right)^{2}
$$

Often, we only have a sample of data. For example, we might want to estimate the variance in height for students at a university using just a random sample of students. 
When using a sample from a larger data set to estimate the variance, the above formula has a *bias*. Therefore, it is common to use the *unbiased* estimator

$$
s^{2} = \left( \frac{\sum_{i=0}^{n-1} x^{2}_{i}}{n} - \left( \frac{ \left( \sum_{i=0}^{n-1} x_{i} \right)}{n} \right)^{2} \right) \frac{n}{n-1}
$$

to estimate the variance. In this exercise we will use the unbiased estimator.

1. Create a function that returns the estimated variance for a list of numbers based on the above equation. Test your function using 1 million samples drawn from a Gaussian distribution with a mean of 10 and a standard deviation of 3. Use your crsid to seed the random number generator
   
   *See hint below on how to create the sample.*

2. For a sample drawn from a distribution with mean $5 \times 10^6$ and standard deviation $2.0$, estimate the variance using (i) your function for estimating the variance, and (ii) using the `variance` function from the Python `statistics` module. Comment on and explain any significant differences in the results from (i) and (ii).

### Hint: sampling from a distribution

The function `random.gauss` can be used to sample a Gaussian distribution with a specified mean and standard deviation (square root of the variance) *N* times, e.g.:

In [154]:
import random

random.seed("gnw20")  # See the random number generator

mu = 10.0    # mean of the distribution
sigma = 3.0  # Standard deviation
x = [random.gauss(mu, sigma) for i in range(8)]
print(x)

[11.176693418313244, 12.08792069683015, 12.649052763696117, 9.845742673456446, 7.317196693264903, 6.876099163343957, 15.181269044169637, 12.844967463974331]


### Solution

1. Estimate variance

In [160]:
import random
import math 
import statistics

random.seed("sc2664") # seeding with my crsid

# implementing above formula 
def estimate_variance(x):

    N = len(x)

    # obtaining sums 
    sample_sum = 0
    squared_sample_sum = 0
    
    for i in range(len(x)):
        sample_sum += x[i]
        squared_sample_sum += x[i]**2
    
    variance = (N/(N-1)) * (squared_sample_sum/N - (sample_sum/N)**2)

    return variance

# testing with mu = 10, sigma = 3, N = 100
mu = 10 # mean
sigma = 3 # S.D. 
n = 1000 # number of samples drawn from Gaussian distribution 

x = [random.gauss(mu, sigma) for i in range(n)]
my_variance = estimate_variance(x)

print(my_variance)

8.868655012783572


2. Use the `statistics` module to estimate the variance, and compare the estimated variance using `statistics` to the estimated variance using your implementation for the variance estimation.

In [164]:
py_variance = statistics.variance(x)

print(f"python module variance: {py_variance}")
print(f"my estimated variance: {my_variance}")
print(f"difference: {abs(my_variance-py_variance)}")

# my estimation is consistently lower, however the difference is not significant (11th decimal point). 
# the difference could be due to the fact that I used an unbiased estimator

python module variance: 8.86865501278378
my estimated variance: 8.868655012783572
difference: 2.078337502098293e-13


## Exercise 05.03 (optional, parallel processing)

Almost all modern computer processing units have multiple processing *cores*. To utilise the full performance of a processor, operations need to be performed in *parallel*, i.e. each processing core is given a task to perform.

Parallel computing is a very rich and technical area. To help exploit multi-core systems there are libraries that support parallel processing. Below is a simple example using the Python `multiprocessing` library.

In [ ]:
import multiprocessing

# On some operaring systems, you may need to uncomment the below line
# multiprocessing.set_start_method('fork')

def f(task):
    """A function that print the input argument and the id for the process that executes the function"""
    print(f"Task index {task}, process id: {multiprocessing.current_process()}\n")
     
    return str(multiprocessing.current_process())
        
        
# Using 3 'processes', execute the function 'f' four times (each time with a different argument)  
with multiprocessing.Pool(processes=3) as p:
    procs = p.map(f, [0, 1, 2, 3])  # Call function mysort three times
    
    print("Returned data (a list)") 
    print(procs)

Consider the below code that creates a list of lists of sorted numbers.

In [ ]:
def mysort(N):
    """Create a randomly ordered list of integers of length N, and return the sorted list"""
    # Create randomly ordered list
    x = random.sample(range(0, N), N)
 
    # Return sorted list of numbers
    return sorted(x)

def sorted_lists(N, p):
    """Create a list of sorted lists"""
    data = []
    for i in range(p):
        data.append(mysort(N))

%time x = sorted_lists(1000000, 5)

Use the `multiprocessing` module to perform the above operation in parallel. Investigate how the processing time changes with the number of processes, and in particular the average time per 'create and sort operation' when changing the number of processes.

In [ ]:
...